# Generation checks

Full pipeline test from a raw guest question through to a generated answer, with an LLM doing
the query understanding as well as the final reply -- two calls per question, both fully
printed:

1. **Understand** -- one structured-output LLM call reads the question and returns intent
   (`menu` vs `faq`) plus retrieval filters (dietary, price ceiling, allergens to exclude).
   This replaces `01_retrieval_checks.ipynb`'s regex-based `parse_constraints()` and this
   notebook's earlier top-1-`item_type` `classify_intent()` -- both were explicitly called out
   as heuristics to swap for an LLM call once one was available.
2. **Retrieve** -- the rest of `01`'s pipeline, unchanged: hybrid `alpha=0.75` -> allergen
   exclude (union of `allergens_contains`/`allergens_may_contain`) -> rerank `rerank-v3.5` ->
   gate `0.15`.
3. **Generate** -- a grounding-only system prompt, with a tone instruction picked from the
   understanding step's intent: **precise and literal for menu facts**
   (price/allergens/nutrition must stay exact), **warm and conversational for FAQ answers**
   (house-policy copy can be phrased more naturally). Unlike Gemini's 3.x Flash line (this
   notebook's previous backend, which silently ignored `temperature`/`top_p`/`top_k`), Groq's
   models genuinely respect `temperature` -- confirmed directly (same prompt at `temperature=0`
   came back identical three times in a row; at `temperature=1.8` it varied every time). Section
   5 now pairs the tone instruction with a real per-intent `temperature`, rather than using
   prompt text as a `temperature` substitute.

**Runs the app's own code (2026-09-25).** Everything between the guest's message and the
answer that does not call an LLM is imported from `app/`, not copied: the understanding prompt,
schema and parsing (`app/agent/understanding.py`), menu browsing and card clicks
(`app/agent/browse.py`), retrieval (`app/retrieval.py`'s `RetrievalTool`: filters, hybrid
search, allergen exclusion, rerank, gate, relaxation, the "a filter removed this dish" NOTE),
the tone and temperature, prompt assembly including what a follow-up refers back to, the
streaming decoder, the leak check and the reply parse (`app/agent/generation.py`), the item
cards (`app/agent/cards.py`), the per-stage timing (`app/timing.py`) and every system prompt
(`app/agent/prompts.py`). What this notebook still defines itself: the LLM transport (it also
speaks Gemini, and keeps Groq's server-side timings), `generate_answer()` and `answer()` (the
app's graph nodes as plain functions, with every intermediate step returned), and the
printouts. `generate_answer()` and `answer()` follow `app/agent/graph.py` step for step.
The Groq-only pieces (SSE streaming, `reasoning_effort`) degrade gracefully under
`LLM_PROVIDER=gemini`: the reply arrives as one piece, so its time to first token equals its
total generate time. Understanding also **routes**: a greeting, an off-topic message or
menu browsing ("what is your menu?", "show me the drinks") is answered directly -- from a fixed
reply or from the menu catalog, with no search and no generation call (sections 3 and 8).

Every test cell prints both full prompts (understanding and generation) and both LLM outputs,
so the whole path from question to answer is inspectable, not just the final text.

Runs against the live Weaviate collection and the live LLM API -- nothing is mocked.

Prerequisites: the collection is loaded (`scripts/load_knowledge_base.py`), and `.env` holds
`WEAVIATE_URL` / `WEAVIATE_API_KEY`, `EMBEDDING_API_KEY` (Cohere, for reranking), and
`GROQ_API_KEY` or `LLM_API_KEY` (Gemini), matching `LLM_PROVIDER` (`groq` by default; see
section 2) -- `00_environment_check.ipynb` verifies the Groq path. Reranking needs the Cohere key;
a trial key is rate-limited, so some cells may fall back to hybrid order (see `01`). Without
an LLM key configured for the active provider, the understanding step falls back to `{intent: "menu", no filters}` and
generation is skipped, so retrieval-only checks still run.

## Pipeline overview

What happens between a guest's message and the answer -- the same steps, in the same order, as
the app's LangGraph agent (`app/agent/graph.py`: query -> respond, or query -> retrieve ->
ground -> answer). At most two LLM calls (provider per `LLM_PROVIDER`, section 2) and up to two
Cohere calls per question per search attempt (the embedding call is made by Weaviate itself,
mid-retrieval, and isn't shown as a separate box the way the direct rerank call is -- see section
4's note on why it can't be measured from here).

Node names are the functions each step calls. Every one of them is the app's own code, imported
from `app/`, except the two LLM calls (`call_llm()` / `stream_llm()`, this notebook's
provider-switchable transport) and `answer()` / `generate_answer()`, which run the app's graph
nodes as plain functions.

```mermaid
flowchart TD
    Q[Guest message] --> CLICK{"card click?<br/>answer(browse=...)"}
    CLICK -- "yes: a known group or category" --> PICK["picked_browse()<br/>no LLM call"]
    CLICK -- "no: typed text or a dish card" --> U["understand_query()<br/>LLM 1 (LLM_PROVIDER)"]
    H["last 3 turns (optional)<br/>build_history_context()"] -.-> U
    U --> POST["parse_understanding()<br/>drops unknown values, turns a browse stating a requirement<br/>or an off_topic naming a real dish into menu, adds resolved_question"]
    POST --> RT{"intent"}
    PICK --> DR
    RT -- "greeting, off_topic, menu_browse" --> DR["direct_answer()<br/>fixed reply or the catalog<br/>no search, no generation call"]
    DR --> DCARDS["picture cards for a list of groups or categories,<br/>item cards for a category's items"]
    DCARDS --> ANS

    RT -- "menu, faq" --> F
    subgraph SEARCH["RetrievalTool.search()"]
        F["build_filter()<br/>build_search_text()"] --> R["retrieve()<br/>Weaviate hybrid, top 20"]
        R --> EX["drop rows whose contains or<br/>may_contain hits an excluded allergen"]
        EX --> RR["rerank()<br/>Cohere, keep top 6"]
        RR --> GATE{"top score >= 0.15?"}
        GATE -- "no, and a relaxable filter is set" --> RELAX["drop one filter:<br/>kcal, protein, gluten-free,<br/>alcohol-free, then price"]
        RELAX --> F
        GATE -- "yes, or nothing left to relax" --> LOOK["diet or allergen filter set?<br/>unfiltered top-1 lookup:<br/>did a filter remove the named dish?"]
    end

    LOOK --> CTX["build_context()<br/>or no confident match"]
    CTX --> PROMPT["build_user_prompt()<br/>guest message, what it refers back to,<br/>CONTEXT, relaxation and exclusion NOTEs"]
    PROMPT --> TEMP["tone_for() / temperature_for()<br/>+ citation format if a dish can be cited"]
    TEMP --> GEN["stream_llm()<br/>LLM 2 (LLM_PROVIDER)"]
    GEN --> GUARD["AnswerStreamDecoder + LeakHoldback<br/>parse_generation_reply(),<br/>salvage_cited_slugs() if the JSON broke"]
    GUARD --> CARDS["cards_for_answer()<br/>every retrieved dish the answer names or cites,<br/>never one a filter screened out"]
    CARDS --> ANS["Answer + item cards<br/>turn added to history"]
```

Every box is timed with `timed()` (section 1.1), under the same stage names the app logs.

## 1. Connect

Loads `.env`, connects to Weaviate Cloud (same as `01`), and reads the LLM credentials
(`00_environment_check.ipynb` verifies these are valid). The client stays open for the whole
notebook, including the demo cell at the end -- re-run this cell to reconnect if the kernel
session drops. Closes any `client` left over from a previous run of this same cell first --
otherwise reconnecting in an already-running kernel leaks the old connection's sockets until
Python's garbage collector eventually finalizes them, which is what an "unclosed
`<ssl.SSLSocket ...>`" `ResourceWarning` after a reconnect actually is.

In [ ]:
import contextlib
import json
import os
import random
import time

import httpx
import weaviate
from dotenv import find_dotenv, load_dotenv
from weaviate.classes.init import AdditionalConfig, Auth, Timeout

load_dotenv(find_dotenv(usecwd=True))

PROVIDER = os.environ.get("EMBEDDING_PROVIDER", "cohere").lower()
COHERE_KEY = os.environ.get("EMBEDDING_API_KEY", "")
_hdr = "X-OpenAI-Api-Key" if PROVIDER == "openai" else "X-Cohere-Api-Key"

# Re-running this cell in an already-running kernel would otherwise leave the previous
# client's sockets open until Python's garbage collector gets around to them -- that's what
# an "unclosed <ssl.SSLSocket ...>" ResourceWarning after a reconnect is. Close it first.
# globals().get(...) (rather than referencing `client` directly) means this doesn't depend on
# `client` already existing in this kernel session.
_previous_client = globals().get("client")
if _previous_client is not None:
    with contextlib.suppress(Exception):
        _previous_client.close()

client = weaviate.connect_to_weaviate_cloud(
    cluster_url=os.environ["WEAVIATE_URL"],
    auth_credentials=Auth.api_key(os.environ["WEAVIATE_API_KEY"]),
    headers={_hdr: COHERE_KEY},
    # Explicit timeouts, as in app/retrieval.py: a stalled Weaviate call fails fast (and
    # shows up in the `weaviate` timing) instead of hanging a turn for the client default.
    additional_config=AdditionalConfig(timeout=Timeout(init=5, query=15)),
)
kb = client.collections.get("KnowledgeBase")
print("connected -", kb.aggregate.over_all(total_count=True).total_count, "objects")

# LLM_PROVIDER selects which of the two call_llm() code paths below actually runs -- "groq"
# (OpenAI-compatible chat/completions) or "gemini" (generateContent). Edit directly, or set
# LLM_PROVIDER in .env, to switch. See section 2's markdown for why each needs its own path
# rather than one shared request shape.
LLM_PROVIDER = os.environ.get("LLM_PROVIDER", "groq").lower()
if LLM_PROVIDER != "gemini":
    # Anything but "gemini" runs the Groq path -- the app always uses Groq and ignores this
    # setting -- so call it "groq": RATE_LIMITS and the key pool are keyed by that name.
    if LLM_PROVIDER != "groq":
        print(f"note: LLM_PROVIDER={LLM_PROVIDER!r} is not 'gemini', so Groq is used, as in app/")
    LLM_PROVIDER = "groq"

if LLM_PROVIDER == "gemini":
    LLM_API_KEY = os.environ.get("LLM_API_KEY", "")
    UNDERSTAND_MODEL = os.environ.get("LLM_MODEL", "gemini-3.8-flash")
    GENERATION_MODEL = UNDERSTAND_MODEL
else:
    LLM_API_KEY = os.environ.get("GROQ_API_KEY", "")
    UNDERSTAND_MODEL = os.environ.get("UNDERSTAND_MODEL", "openai/gpt-oss-120b")
    GENERATION_MODEL = os.environ.get("GENERATION_MODEL", "openai/gpt-oss-120b")

print(f"LLM_PROVIDER     = {LLM_PROVIDER}")
print(f"UNDERSTAND_MODEL = {UNDERSTAND_MODEL}")
print(f"GENERATION_MODEL = {GENERATION_MODEL}")
_key_status = f"set ({len(LLM_API_KEY)} chars)" if LLM_API_KEY else "not set"
# Not the last word on whether the LLM cells run: the numbered pool keys count too, and are read
# in section 2.1, which sets LLM_AVAILABLE.
print(f"LLM_API_KEY      = {_key_status}")

## 1.1 Latency instrumentation

Per-stage timing with `track_timings()` / `timed()` imported from `app/timing.py` (the app's
retrieval code records into the same per-turn dict), and the same stage names (`turn`, `understand`, `weaviate`, `rerank`, `rerank_pace`, `generate`,
`llm_backoff`, `first_delta`), so a figure here means what it means in the app's
`chat turn timings` log line and in `scripts/latency_check.py --log`. Outside a
`track_timings()` block `timed()` does nothing, so it costs nothing in a cell that doesn't ask
for timings. Every `answer()` returns its turn's `timings`, and `show_answer()` prints them.

What is measured -- more than one number, because "how long did it take" has several answers:

- **End to end (`turn`)** and each stage's share of it. Stages nest on purpose: `rerank`
  includes `rerank_pace`, and `understand` / `generate` include any `llm_backoff`.
- **Time to first token.** `first_token` is when the model's first text arrives, measured from
  the start of the generate stage. `first_delta` is when the first word can be *shown to the
  guest*, measured from the start of the turn -- it also pays for understanding, retrieval and
  the few-word leak holdback. In a chat UI `first_delta` is the number that is felt.
- **Streaming speed.** The mean gap between streamed chunks, and output tokens per second over
  the generate stage (reasoning tokens included, so it understates visible words per second).
- **Waiting vs working.** `rerank_pace` (this notebook's own Cohere rate-limit pacer) and
  `llm_backoff` (retry sleeps after a 429/5xx) are time spent *not* working. `work` is `turn`
  minus both, so a slow turn that is mostly waiting reads as a quota problem, not a speed one.
- **Tails, not averages; cold apart from warm.** `latency_report()` prints n / mean / p50 / p95
  / max per stage over many turns. `benchmark_latency()` reports its first turn(s) as *cold*
  (they pay DNS, TCP+TLS and connection-pool setup) and only the rest in the percentile table.
  With fewer than ~20 turns p95 is effectively the max.
- **Clock and provider time.** Everything uses `time.perf_counter()` (monotonic, highest
  resolution) rather than `time.time()`. Where Groq's usage block reports its own server-side
  durations (queue / prompt / completion time), `show_answer()` prints them too, which separates
  the provider's time from the network's.

In [ ]:
from collections.abc import Iterator

from app.agent.memory import HistoryTurn
from app.timing import _active, timed, track_timings

# Per-turn latency accounting is app/timing.py itself, imported rather than copied: one dict per
# turn held in a ContextVar, a no-op outside a turn, repeated stages accumulate. It has to be the
# app's own ContextVar, not a copy: the app code this notebook runs (RetrievalTool's `weaviate`,
# `rerank` and `rerank_pace` stages) records into that one, and so do mark() and record() below.
# Stage names are the app's too, so a number read here means what the same key means in the
# app's "chat turn timings" log line (and in scripts/latency_check.py --log).


def mark(stage: str, since: float) -> None:
    """Record the ms elapsed since `since` (a perf_counter reading) under `stage` -- first call
    only, so it captures a "time to first ..." event rather than the latest one."""
    timings = _active.get()
    if timings is not None and stage not in timings:
        timings[stage] = (time.perf_counter() - since) * 1000


def record(stage: str, value: float) -> None:
    """Add a measurement taken elsewhere (e.g. a chunk count) to this turn's timings."""
    timings = _active.get()
    if timings is not None:
        timings[stage] = timings.get(stage, 0.0) + value


# The metrics below are the ones LLM-serving latency is conventionally judged on, not a single
# wall-clock number:
#   * end-to-end latency  -- `turn`, and each stage's share of it;
#   * time to first token -- `first_token` (the model's first streamed text, measured from the
#     start of the generate stage) and `first_delta` (the first word the GUEST can see, measured
#     from the start of the turn -- it also pays for understand + retrieval + the leak holdback);
#   * streaming speed     -- the mean gap between streamed chunks, and output tokens per second;
#   * waiting vs working  -- `rerank_pace` (our own Cohere pacer) and `llm_backoff` (retry sleeps
#     after a 429/5xx) are time spent NOT working, so `work` = `turn` - both. A slow turn that is
#     mostly waiting is a quota problem, not a speed problem;
#   * tails, not averages -- p50/p95 over many turns (`latency_report`), with warm-up turns
#     reported separately because the first request pays connection setup (DNS, TCP, TLS).
WORK_STAGES = ("understand", "weaviate", "rerank", "generate")


def derive_metrics(timings: dict[str, float]) -> dict[str, float]:
    """`timings` plus the derived figures: wait, work, and the glue outside every stage."""
    out = dict(timings)
    turn = timings.get("turn")
    if turn is None:
        return out
    # A turn that never waited waited 0 ms -- record that rather than leave the stage absent.
    out.setdefault("rerank_pace", 0.0)
    out.setdefault("llm_backoff", 0.0)
    out["wait"] = out["rerank_pace"] + out["llm_backoff"]
    out["work"] = turn - out["wait"]
    out["other"] = turn - sum(timings.get(stage, 0.0) for stage in WORK_STAGES)
    stream_gaps = timings.get("gen_chunks", 0.0) - 1
    if stream_gaps > 0 and "stream" in timings:
        out["chunk_gap"] = timings["stream"] / stream_gaps
    return out


def format_timings(timings: dict[str, float], usage: dict | None = None) -> str:
    """One turn's latency as a readable table: stages, waiting vs working, first-token times."""
    t = derive_metrics(timings)
    turn = t.get("turn")
    if turn is None:
        return "(no timings recorded)"

    def row(label: str, key: str, indent: int = 1) -> str | None:
        if key not in t:
            return None
        share = f"{100 * t[key] / turn:5.0f}%" if turn else ""
        return f"{'  ' * indent}{label:<36}{t[key]:9,.0f} ms  {share}"

    lines = [
        f"{'turn (end to end)':<38}{turn:9,.0f} ms",
        row("understand (LLM 1)", "understand"),
        row("weaviate (hybrid query)", "weaviate"),
        row("rerank (Cohere)", "rerank"),
        row("generate (LLM 2, streamed)", "generate"),
        row("other (prompt build, glue)", "other"),
        "waiting rather than working:",
        row("rerank pacer sleep", "rerank_pace"),
        row("LLM retry backoff", "llm_backoff"),
        row("=> working time (turn - waiting)", "work"),
        "what the guest feels:",
        row("first token from the model", "first_token"),
        row("first visible word (turn start)", "first_delta"),
    ]
    if "chunk_gap" in t:
        lines.append(
            f"  streaming: {t['gen_chunks']:.0f} chunks, mean gap {t['chunk_gap']:.1f} ms"
            f" between chunks"
        )
    completion = (usage or {}).get("generate", {}).get("completion_tokens", 0)
    if completion and t.get("generate"):
        lines.append(
            f"  output speed: {completion / (t['generate'] / 1000):.0f} tokens/s over the whole"
            f" generate stage ({completion} completion tokens, reasoning included)"
        )
    return "\n".join(line for line in lines if line)


def percentile(values: list[float], q: float) -> float:
    """The q-th percentile (0-100) by linear interpolation between the closest ranks."""
    ordered = sorted(values)
    if not ordered:
        raise ValueError("percentile() of an empty list")
    pos = (len(ordered) - 1) * q / 100
    low = int(pos)
    high = min(low + 1, len(ordered) - 1)
    return ordered[low] + (ordered[high] - ordered[low]) * (pos - low)


REPORT_STAGES = [
    "turn",
    "first_delta",
    "first_token",
    "understand",
    "weaviate",
    "rerank",
    "generate",
    "rerank_pace",
    "llm_backoff",
    "work",
    "wait",
]


def latency_report(samples: list[dict[str, float]], title: str = "latency") -> None:
    """Print n / mean / p50 / p95 / max per stage over many turns' timings (ms).

    With few samples p95 is close to the max -- read the n column before trusting a tail.
    """
    derived = [derive_metrics(s) for s in samples if "turn" in s]
    if not derived:
        print(f"{title}: no timed turns yet")
        return
    print(f"{title} -- {len(derived)} turn(s), milliseconds")
    print(f"  {'stage':<14}{'n':>4}{'mean':>9}{'p50':>9}{'p95':>9}{'max':>9}")
    for stage in REPORT_STAGES:
        values = [d[stage] for d in derived if stage in d]
        if not values:
            continue
        print(
            f"  {stage:<14}{len(values):>4}{sum(values) / len(values):>9,.0f}"
            f"{percentile(values, 50):>9,.0f}{percentile(values, 95):>9,.0f}{max(values):>9,.0f}"
        )
    if len(derived) < 20:
        print("  (fewer than 20 turns: p95 is effectively the max)")


def benchmark_latency(
    questions: list[str], runs: int = 3, warmup: int = 1, history: list[HistoryTurn] | None = None
) -> list[dict[str, float]]:
    """Ask `questions` `runs` times over and report latency, warm-up turns kept apart.

    The first `warmup` turns pay one-off costs (DNS, TCP+TLS to Groq/Cohere/Weaviate, a cold
    connection pool), so they are reported as "cold" and left out of the percentile table.
    Makes 2 LLM calls + 1-2 Cohere rerank calls per turn against the live APIs, and Cohere's
    pacer alone spaces turns by 60 / COHERE_MAX_RPM seconds -- keep `runs` small.
    """
    cold: list[dict[str, float]] = []
    warm: list[dict[str, float]] = []
    failures = 0
    for i, question in enumerate(questions * runs):
        try:
            r = answer(question, history=history)
        except Exception as e:  # one failed turn must not lose the ones already measured
            failures += 1
            print(f"   [turn {i + 1} failed: {type(e).__name__}: {e}]")
            continue
        (cold if i < warmup else warm).append(r["timings"])
    if cold:
        print(
            "cold (warm-up) turns, ms: "
            + ", ".join(f"{s['turn']:,.0f}" for s in cold)
            + "  -- includes one-off connection setup"
        )
    latency_report(warm, "warm turns")
    if failures:
        print(f"{failures} turn(s) failed and are not counted")
    return warm

## 2. LLM call helper

One thin wrapper around whichever provider `LLM_PROVIDER` selects -- Groq's OpenAI-compatible
`chat/completions` endpoint, or Gemini's `generateContent` REST endpoint (same style
`00_environment_check.ipynb` uses for its credential check) -- shared by both LLM calls this
notebook makes: query understanding (structured JSON, via `response_schema`) and answer
generation (free text). Keeping one call site per concern means both share the same error
handling, timeout, retry, and rotation behavior regardless of which provider is active,
instead of drifting apart the way this notebook and its Gemini-only predecessor once did.

The two providers genuinely differ, not just in URL/auth shape:

- **Groq** genuinely applies `temperature` (confirmed directly: `temperature=0` is
  deterministic, higher values vary) and supports `reasoning_effort` (`"low"`/`"medium"`/
  `"high"`, gpt-oss models only, controlling how many reasoning tokens the model spends before
  answering).
- **Gemini**'s 3.x Flash line silently ignores `temperature`/`top_p`/`top_k` and has no
  `reasoning_effort` equivalent -- the API accepts them, returns `200`, and does nothing with
  them (Google's own confirmed behavior). When `LLM_PROVIDER == "gemini"`, `call_llm()` drops
  both parameters rather than sending something the provider silently no-ops; section 5's tone
  instruction is what actually steers Gemini's output instead.

`call_llm()` returns `{"text": ..., "usage": {...}}` either way -- both providers' own token
counts (Groq's `usage`, Gemini's `usageMetadata`) are read directly rather than estimated from
string length, which is what lets `show_answer()` (section 8) report exactly how many tokens
each question consumed.

Two layers of rate-limit handling, not just one, regardless of provider:

- **Proactive pacing** -- `_pace_llm_call()` blocks just long enough before every request to
  keep the call rate under `LLM_MAX_RPM`, which is itself provider-dependent (Groq's
  confirmed per-key quota is far higher than Gemini's free-tier per-minute cap -- see the code
  cell below). Edit the constant directly to match your key's actual quota.
- **Reactive retry**, for whatever pacing doesn't prevent (a burst from another process on the
  same key, a transient 5xx, a dropped connection): 429/5xx/connection errors retry with
  full-jitter exponential backoff, honoring a `Retry-After` header when the API sends one. A
  non-retryable error (400 bad request, 401/403 auth) raises immediately.

**What comes from `app/agent/llm.py`.** The retry policy (`RETRYABLE_HTTP_CODES`,
`MAX_LLM_RETRIES`, the backoff bounds), Groq's URL, the HTTP timeout, the free-tier limits,
the error types and the key-pool loader are imported from the app. The transport below stays
this notebook's own because the app's `GroqClient` speaks Groq only; it follows `GroqClient`
step for step.

**Transport, kept in step with `app/agent/llm.py`.** Every request goes through one
shared keep-alive `httpx.Client` instead of a fresh TCP+TLS handshake per call, and
`stream_llm()` returns the reply as it is generated (Groq's server-sent events) instead of
waiting for all of it -- how answer generation runs, and what makes a *time to first token*
measurable. Gemini has no streamed path here, so `stream_llm()` hands its whole reply back
as a single piece. Retry sleeps are timed as `llm_backoff` (section 1.1), so time spent
waiting out a 429 shows up apart from the call itself.

In [ ]:
from app.agent.llm import (
    BASE_DELAY_S,
    GROQ_RPD_LIMIT,
    GROQ_RPM_LIMIT,
    GROQ_URL,
    LLM_HTTP_TIMEOUT,
    MAX_DELAY_S,
    MAX_LLM_RETRIES,
    RETRYABLE_HTTP_CODES,
    AllKeysRateLimitedError,
    LLMStreamError,
    Usage,
    _http_error_detail,
    load_groq_key_pool,
    zero_usage,
)
from app.agent.llm import LLM_MAX_RPM as GROQ_MAX_RPM

# The retry policy, Groq's URL and timeout, the free-tier limits, the error types and the key-pool
# loader are app/agent/llm.py's own, imported rather than copied. What stays in this cell is the
# transport, because it does two things the app's GroqClient doesn't: it also speaks Gemini's API
# (LLM_PROVIDER=gemini), and its LLMStream keeps the server-side durations Groq reports
# (`provider_timings`, printed by show_answer()). Its retry, pacing and key rotation follow
# GroqClient's step for step.
LLM_MAX_RPM = GROQ_MAX_RPM if LLM_PROVIDER == "groq" else 15.0

# Free-tier request limits, confirmed 2026-09-18 -- a *local, proactive* guard, separate from
# LLM_MAX_RPM's reactive pacing above. RATE_LIMITS backs _key_available()/_record_key_usage()
# in the key-pool cell below: a pool key already at its own confirmed limit is skipped with no
# HTTP request made, rather than tried and left to 429. Groq's figures are the app's.
RATE_LIMITS = {
    "groq": {"rpm": GROQ_RPM_LIMIT, "rpd": GROQ_RPD_LIMIT},
    "gemini": {"rpm": 15, "rpd": 1500},
}

_last_llm_call_at = 0.0

# Cloudflare (in front of api.groq.com) 403s a bare library-default User-Agent (Python's
# "Python-urllib/x.y", httpx's "python-httpx/x.y") with body "error code: 1010" -- easy to
# mistake for an auth failure. A normal-looking User-Agent avoids it.
_GROQ_HEADERS = {
    "Content-Type": "application/json",
    "User-Agent": "Mozilla/5.0 (compatible; wagami-rag-notebook/1.0)",
}

# One shared keep-alive connection pool for every LLM request instead of a fresh TCP+TLS
# handshake per call -- the same choice app/agent/llm.py makes (~125ms saved per call to
# api.groq.com, measured there). The read timeout also bounds the gap between streamed chunks,
# not just a whole response. Re-running this cell closes the previous pool first.
_previous_llm_http = globals().get("_llm_http")
if _previous_llm_http is not None:
    with contextlib.suppress(Exception):
        _previous_llm_http.close()
_llm_http = httpx.Client(timeout=LLM_HTTP_TIMEOUT, headers=_GROQ_HEADERS)


def _pace_llm_call() -> None:
    """Block just long enough to keep the LLM calls under LLM_MAX_RPM requests/minute."""
    global _last_llm_call_at
    min_interval = 60.0 / LLM_MAX_RPM
    wait = min_interval - (time.monotonic() - _last_llm_call_at)
    if wait > 0:
        time.sleep(wait)
    _last_llm_call_at = time.monotonic()


def _build_llm_request(
    api_key: str,
    system_prompt: str,
    user_prompt: str,
    model: str,
    response_schema: dict | None,
    temperature: float | None,
    reasoning_effort: str | None,
    *,
    stream: bool,
) -> tuple[str, bytes, dict[str, str]]:
    """(url, body, extra headers) for one request, against whichever provider LLM_PROVIDER
    selects. Gemini is never streamed here -- see _stream_llm_single_key()."""
    if LLM_PROVIDER == "gemini":
        url = (
            f"https://generativelanguage.googleapis.com/v1beta/models/{model}:generateContent"
            f"?key={api_key.strip()}"
        )
        generation_config: dict = {}
        if response_schema is not None:
            generation_config["responseMimeType"] = "application/json"
            generation_config["responseSchema"] = response_schema
        body = json.dumps(
            {
                "systemInstruction": {"parts": [{"text": system_prompt}]},
                "contents": [{"role": "user", "parts": [{"text": user_prompt}]}],
                "generationConfig": generation_config,
            }
        ).encode()
        return url, body, {}
    payload: dict = {
        "model": model,
        "messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
    }
    if temperature is not None:
        payload["temperature"] = temperature
    if reasoning_effort is not None:
        payload["reasoning_effort"] = reasoning_effort
    if response_schema is not None:
        payload["response_format"] = {
            "type": "json_schema",
            "json_schema": {"name": "response", "strict": True, "schema": response_schema},
        }
    if stream:
        payload["stream"] = True
        payload["stream_options"] = {"include_usage": True}
    return GROQ_URL, json.dumps(payload).encode(), {"Authorization": f"Bearer {api_key.strip()}"}


def _llm_request_with_retries(
    url: str, body: bytes, headers: dict[str, str], *, stream: bool, max_retries: int
) -> httpx.Response:
    """POST `body`, retrying retryable failures with jittered exponential backoff.

    Returns the first successful response -- fully read for stream=False, headers-only (body
    still unread, caller must close) for stream=True. Raises httpx.HTTPStatusError for a
    non-retryable status or once retries run out. A timeout is never retried: a 60s read
    timeout retried five times would hold a guest for minutes. Retry sleeps are timed as
    `llm_backoff` (section 1.1) so waiting on a 429 shows up apart from the call itself.
    """
    delay_cap = BASE_DELAY_S
    for attempt in range(1, max_retries + 1):
        _pace_llm_call()
        request = _llm_http.build_request("POST", url, content=body, headers=headers)
        retry_after: str | None = None
        detail = ""
        try:
            response = _llm_http.send(request, stream=stream)
        except httpx.TimeoutException:
            raise
        except httpx.TransportError as e:
            label = f"connection error ({e})"
            if attempt == max_retries:
                raise
        else:
            if response.is_success:
                return response
            response.read()
            response.close()
            retryable = response.status_code in RETRYABLE_HTTP_CODES
            detail = _http_error_detail(response)
            if not retryable or attempt == max_retries:
                print(f"   [LLM HTTP {response.status_code}, giving up -- {detail}]")
                response.raise_for_status()
            retry_after = response.headers.get("Retry-After")
            label = f"HTTP {response.status_code} {response.reason_phrase}"

        if retry_after:
            try:
                wait = float(retry_after)
            except ValueError:
                wait = random.uniform(0, delay_cap)
        else:
            wait = random.uniform(0, delay_cap)
        suffix = f"  {detail}" if detail else ""
        print(f"   [LLM {label}; retrying in {wait:.1f}s ({attempt}/{max_retries})]{suffix}")
        with timed("llm_backoff"):
            time.sleep(wait)
        delay_cap = min(delay_cap * 2, MAX_DELAY_S)

    raise RuntimeError("unreachable")  # the loop above always returns or raises


def _parse_llm_completion(response: httpx.Response) -> dict:
    """{"text", "usage"} from a fully-read, non-streamed response of either provider."""
    data = response.json()
    if LLM_PROVIDER == "gemini":
        parts = data["candidates"][0]["content"]["parts"]
        text = " ".join(p.get("text", "") for p in parts).strip()
        meta = data.get("usageMetadata", {})
        usage = {
            "prompt_tokens": meta.get("promptTokenCount", 0),
            "completion_tokens": meta.get("candidatesTokenCount", 0),
            "total_tokens": meta.get("totalTokenCount", 0),
        }
    else:
        text = data["choices"][0]["message"]["content"].strip()
        meta = data.get("usage", {})
        usage = {
            "prompt_tokens": meta.get("prompt_tokens", 0),
            "completion_tokens": meta.get("completion_tokens", 0),
            "total_tokens": meta.get("total_tokens", 0),
        }
    return {"text": text, "usage": usage}


class LLMStream:
    """One live streamed chat completion: iterate it for text deltas as they arrive.

    Same as app/agent/llm.py's LLMStream, plus `provider_timings`. `usage` is only complete
    once iteration has finished; it stays zero if the stream is abandoned early. Always closes
    the underlying connection when iteration ends -- normally, on error, or when the consumer
    stops early.
    `provider_timings` holds whatever server-side durations (seconds) Groq's final usage block
    reports (queue_time, prompt_time, completion_time, total_time), if it reports any -- the
    part of a slow call that is the provider's, as opposed to the network's.
    """

    def __init__(self, response: httpx.Response) -> None:
        self._response = response
        self.usage: Usage = zero_usage()
        self.provider_timings: dict[str, float] = {}

    def __iter__(self) -> Iterator[str]:
        try:
            for line in self._response.iter_lines():
                if not line.startswith("data:"):
                    continue
                data = line[5:].strip()
                if data == "[DONE]":
                    break
                chunk = json.loads(data)
                if "error" in chunk:
                    raise LLMStreamError(str(chunk["error"])[:400])
                # Groq reports usage on the final chunk under its own `x_groq` key; the
                # OpenAI-style top-level `usage` is read too in case that changes.
                meta = chunk.get("usage") or chunk.get("x_groq", {}).get("usage")
                if meta:
                    self.usage = {
                        "prompt_tokens": meta.get("prompt_tokens", 0),
                        "completion_tokens": meta.get("completion_tokens", 0),
                        "total_tokens": meta.get("total_tokens", 0),
                    }
                    self.provider_timings = {
                        k: float(meta[k])
                        for k in ("queue_time", "prompt_time", "completion_time", "total_time")
                        if isinstance(meta.get(k), int | float)
                    }
                for choice in chunk.get("choices", []):
                    text = choice.get("delta", {}).get("content")
                    if text:
                        yield text
        finally:
            self._response.close()

    def close(self) -> None:
        self._response.close()


class _WholeReplyStream:
    """The stream interface over an already-complete reply (Gemini path): one piece, no SSE."""

    def __init__(self, text: str, usage: Usage) -> None:
        self._text = text
        self.usage = usage
        self.provider_timings: dict[str, float] = {}

    def __iter__(self) -> Iterator[str]:
        if self._text:
            yield self._text

    def close(self) -> None:
        pass


def _call_llm_single_key(
    api_key: str,
    system_prompt: str,
    user_prompt: str,
    model: str,
    response_schema: dict | None,
    temperature: float | None,
    reasoning_effort: str | None,
    max_retries: int,
) -> dict:
    """One non-streamed call for one key -- not called directly by the rest of the notebook;
    call_llm() (key-pool cell below) wraps this with pooling/rotation."""
    url, body, headers = _build_llm_request(
        api_key,
        system_prompt,
        user_prompt,
        model,
        response_schema,
        temperature,
        reasoning_effort,
        stream=False,
    )
    return _parse_llm_completion(
        _llm_request_with_retries(url, body, headers, stream=False, max_retries=max_retries)
    )


def _stream_llm_single_key(
    api_key: str,
    system_prompt: str,
    user_prompt: str,
    model: str,
    temperature: float | None,
    reasoning_effort: str | None,
    max_retries: int,
) -> LLMStream | _WholeReplyStream:
    """One streamed call for one key. Everything that can fail up front -- a 429, a bad key --
    is handled (or raised) here, before any text exists; a failure once the stream is open
    surfaces from iterating it instead, since part of the answer may already be shown.

    Gemini is not streamed by this notebook: it gets the whole reply as a single piece, so its
    time to first token equals its total generate time.
    """
    if LLM_PROVIDER == "gemini":
        reply = _call_llm_single_key(
            api_key, system_prompt, user_prompt, model, None, temperature, None, max_retries
        )
        return _WholeReplyStream(reply["text"], reply["usage"])
    url, body, headers = _build_llm_request(
        api_key,
        system_prompt,
        user_prompt,
        model,
        None,
        temperature,
        reasoning_effort,
        stream=True,
    )
    return LLMStream(
        _llm_request_with_retries(url, body, headers, stream=True, max_retries=max_retries)
    )

### ## 2.1 Key-pool rotation (shared with `03_evaluation.ipynb`)

Both Groq's and Gemini's free tiers cap a single key's request volume enough that iterating on this notebook can burn through a day's quota fast. Reads `GROQ_API_KEY` + `GROQ_API_KEY_1`..`GROQ_API_KEY_21` when
`LLM_PROVIDER == "groq"`, or `LLM_API_KEY` + `LLM_API_KEY1`..`LLM_API_KEY11` when
`LLM_PROVIDER == "gemini"` (same naming each provider's own notebook already used before this
merge -- kept as-is rather than unified, since `.env` files already in use follow these exact
names). `call_llm()` (one-shot: query understanding, the judge) and `stream_llm()` (answer
generation) both run through the same `_dispatch()` rotation, so a streamed call rotates
and skips rate-limited keys exactly like a one-shot one -- the same mechanism as
`GroqClient._dispatch()` in `app/agent/llm.py`.

Each pool key gets exactly one fast attempt (`max_retries=1`)
before rotating -- a 429 is rejected before any generation happens, so it costs no real
tokens, and cycling through all configured keys takes seconds. Only once every key has failed
once does it fall back to one full retry/backoff pass (the full `MAX_LLM_RETRIES`) on the
current key, in case the failure was actually transient rather than the whole pool being
genuinely exhausted.

**Rate-limit awareness, added 2026-09-18:** before trying a key, `_dispatch()`
checks it against `RATE_LIMITS[LLM_PROVIDER]` (Groq: 30 RPM / 1000 RPD; Gemini: 15 RPM / 1500
RPD, both confirmed free-tier figures) via `_key_available()` -- a key already at its own local
limit is skipped with **no HTTP request made**, not tried and left to 429. If every pool key is
at its limit, `call_llm()` raises `AllKeysRateLimitedError` without sending anything at all.
This is a local, approximate, kernel-session-local guard on top of (not instead of) the
provider's own reactive 429 handling above.

In [ ]:
from collections.abc import Callable

if LLM_PROVIDER == "gemini":
    _KEY_POOL = [LLM_API_KEY] if LLM_API_KEY else []
    for _i in range(1, 12):
        _key = os.environ.get(f"LLM_API_KEY{_i}", "").strip()
        if _key:
            _KEY_POOL.append(_key)
else:
    # The app's own loader: GROQ_API_KEY plus GROQ_API_KEY_1.._21, in order.
    _KEY_POOL = load_groq_key_pool(LLM_API_KEY)
print(f"{len(_KEY_POOL)} {LLM_PROVIDER} key(s) available for rotation")

# Whether the LLM cells make real calls. Any key in the pool counts, the unnumbered one or a
# numbered one alone -- the same rule the app uses to decide whether it has a Groq client
# (app/main.py builds one whenever load_groq_key_pool() returns any key).
LLM_AVAILABLE = bool(_KEY_POOL)
if not LLM_AVAILABLE:
    print(
        f"warning: no API key set for LLM_PROVIDER={LLM_PROVIDER!r} -- "
        "understanding/generation cells will use fallbacks"
    )

_key_pool_index = 0

# Per-key fixed-window request counters (RPM + RPD), backing _key_available()/_record_key_usage()
# below -- see RATE_LIMITS (previous cell) for the actual per-provider numbers. Fixed windows,
# not a rolling one -- simpler, and "approximately N requests per minute/day" is the actual
# goal (a client-side safety margin, not exact provider-side parity). Kernel-session-local:
# restarting the kernel resets these, same as _KEY_POOL itself.
_minute_window: dict[str, int] = {}
_minute_count: dict[str, int] = {}
_day_window: dict[str, int] = {}
_day_count: dict[str, int] = {}


def _key_available(key: str) -> bool:
    limits = RATE_LIMITS[LLM_PROVIDER]
    minute = int(time.monotonic() // 60)
    day = int(time.time() // 86400)
    in_minute = _minute_window.get(key) == minute
    in_day = _day_window.get(key) == day
    minute_count = _minute_count.get(key, 0) if in_minute else 0
    day_count = _day_count.get(key, 0) if in_day else 0
    return minute_count < limits["rpm"] and day_count < limits["rpd"]


def _record_key_usage(key: str) -> None:
    minute = int(time.monotonic() // 60)
    day = int(time.time() // 86400)
    if _minute_window.get(key) != minute:
        _minute_window[key] = minute
        _minute_count[key] = 0
    _minute_count[key] += 1
    if _day_window.get(key) != day:
        _day_window[key] = day
        _day_count[key] = 0
    _day_count[key] += 1


def _dispatch[T](attempt: Callable[[str, int], T]) -> T:
    """Run `attempt(key, max_retries)` against the key pool, rotating keys on failure.

    Before ever calling out, each candidate key is checked against its own local RPM/RPD budget
    (_key_available()) -- a key already at its limit is skipped with no HTTP request made. If
    every pool key is at its limit, raises AllKeysRateLimitedError without sending anything.

    With more than one key, each available key gets exactly one fast attempt (max_retries=1, so
    a 429 raises immediately instead of honoring the server's Retry-After) before moving to the
    next -- see this section's markdown for why the naive "let each key retry fully, then
    rotate" version was too slow to use. Falls through to one full retry/backoff call on
    whichever key still has budget once every key has failed once, in case the failure was
    transient rather than the whole pool being genuinely exhausted. Same mechanism as
    app/agent/llm.py's GroqClient._dispatch(); works for streamed and non-streamed calls alike,
    since `attempt` decides what to send.
    """
    global _key_pool_index
    pool = _KEY_POOL or [LLM_API_KEY]
    total_keys = len(pool)
    if total_keys == 1:
        key = pool[0]
        if not _key_available(key):
            raise AllKeysRateLimitedError(
                f"The only configured key is at its local {LLM_PROVIDER} rate limit "
                f"({RATE_LIMITS[LLM_PROVIDER]['rpm']} RPM / {RATE_LIMITS[LLM_PROVIDER]['rpd']} "
                "RPD) -- not sending this request."
            )
        _record_key_usage(key)
        return attempt(key, MAX_LLM_RETRIES)

    for position in range(total_keys):
        idx = _key_pool_index
        key = pool[idx]
        if not _key_available(key):
            print(f"   [key #{idx + 1}/{total_keys} at its local rate limit -- skipping, no call]")
            _key_pool_index = (idx + 1) % total_keys
            continue
        _record_key_usage(key)
        try:
            return attempt(key, 1)
        except httpx.HTTPStatusError:
            _key_pool_index = (idx + 1) % total_keys
            if position < total_keys - 1:
                print(
                    f"   [key #{idx + 1}/{total_keys} failed fast -- rotating to key "
                    f"#{_key_pool_index + 1}/{total_keys}]"
                )

    fallback_key = pool[_key_pool_index]
    if not _key_available(fallback_key):
        fallback_key = next((k for k in pool if _key_available(k)), None)
    if fallback_key is None:
        raise AllKeysRateLimitedError(
            f"All {total_keys} pool key(s) are at their local {LLM_PROVIDER} rate limit "
            f"({RATE_LIMITS[LLM_PROVIDER]['rpm']} RPM / {RATE_LIMITS[LLM_PROVIDER]['rpd']} RPD) "
            "-- not sending this request."
        )
    _record_key_usage(fallback_key)
    print(
        "   [every pool key failed once or was rate-limited -- falling back to full retry/backoff]"
    )
    return attempt(fallback_key, MAX_LLM_RETRIES)


def call_llm(
    system_prompt: str,
    user_prompt: str,
    model: str,
    response_schema: dict | None = None,
    temperature: float | None = None,
    reasoning_effort: str | None = None,
) -> dict:
    """One non-streamed LLM call across the key pool -> {"text", "usage"}. Used for query
    understanding (structured JSON via response_schema) and by the evaluation's judge."""
    return _dispatch(
        lambda key, retries: _call_llm_single_key(
            key,
            system_prompt,
            user_prompt,
            model,
            response_schema,
            temperature,
            reasoning_effort,
            retries,
        )
    )


def stream_llm(
    system_prompt: str,
    user_prompt: str,
    model: str,
    temperature: float | None = None,
    reasoning_effort: str | None = None,
) -> LLMStream | _WholeReplyStream:
    """Like call_llm() for a free-text reply, but returns an open stream to iterate for text
    pieces as they arrive -- how answer generation runs, exactly as in the app."""
    return _dispatch(
        lambda key, retries: _stream_llm_single_key(
            key, system_prompt, user_prompt, model, temperature, reasoning_effort, retries
        )
    )

## 3. Query understanding (LLM)

Replaces two heuristics with one LLM call: `01`'s regex `parse_constraints()` (dietary/price
wording, allergy wording) and this notebook's earlier `classify_intent()` (guessing intent
from whichever row retrieval happened to rank first). `understand_query()` asks the model
directly, before any retrieval happens, for:

- `intent` -- `menu` or `faq`.
- `dietary` -- `vegan`/`vegetarian`/`none`. `none` on purpose for a general availability
  question ("do you have vegan options") -- `01` learned that hard-filtering those loses FAQ
  rows, since FAQ rows carry no `dietary_tags` of their own.
- `price_max_gbp` -- only for a firm ceiling ("under £8"), not vague wording ("affordable").
- `allergens_exclude` -- canonical allergen names, constrained by `response_schema`'s `enum`
  to the same vocabulary `01` used for its regex synonym map, so the model can't return a
  value the corpus wouldn't recognise.
- `search_query` -- the question with dietary/price/allergy wording stripped out, since that's
  already handled by `dietary`/`price_max_gbp`/`allergens_exclude` above. Retrieval and rerank
  use this instead of the raw question (section 4's `search()`), so "vegan" and "under £6"
  stop diluting the vector match for the word that actually identifies the dish.
- `category_hint` -- zero or more of the menu's own category names, guessed from course-type
  language ("starter", "main", "dessert", "drink"). Constrained by `response_schema`'s `enum`
  to `MENU_CATEGORIES`, fetched live from the collection below rather than a hardcoded
  synonym list, so it can't drift from the real menu. `expand_category_hint()` then adds
  sibling categories -- ones sharing an immediate parent in `category_path` -- so getting one
  sibling right pulls in the rest without expecting the model to enumerate every one from
  memory. Folded into the search text as a soft signal (section 4), not a hard filter -- a
  wrong guess should never be able to hide the right dish, only fail to help find it.
- `gluten_free_only` -- true ONLY when the guest asks for the restaurant's own curated
  gluten-free menu/section by name ("what's on your gluten-free menu", "gluten-free options").
  This is a positive filter on `is_gluten_free_listed`, separate from `allergens_exclude`: a
  guest describing an actual allergy ("I'm coeliac", "no gluten please") should still get
  `allergens_exclude` populated too (the safety exclusion), but a guest just browsing the
  named section doesn't need every gluten-containing dish removed from consideration first.
- `kcal_max` -- a number for a calorie ceiling, whether firm ("under 500 calories") or
  qualitative ("a low-calorie main" -> use a sensible reference like 500). null otherwise.
- `protein_min_g` -- a number for a protein floor, whether firm ("at least 20g protein") or
  qualitative ("a high-protein dish" -> use a sensible reference like 20). null otherwise.
  Unlike `price_max_gbp`, qualitative wording is honoured here with a reasonable default
  rather than left null, since "high protein" carries real meaning a guest expects acted on,
  the way "affordable" doesn't for price.
- `alcohol_free` -- true ONLY when the guest explicitly wants a non-alcoholic / alcohol-free
  drink. A positive filter on `abv_percent` being unset.

A live worked example: **"a vegan starter under £6" returned NO CONFIDENT MATCH** the first
time this notebook ran that question, even though the corpus does have exactly one vegan
starter under £6 (an oyster + shiitake mushroom bao bun) -- `data/knowledge_base.json` has no
category literally called "starters", so the raw question's vector match against 39 other
vegan-and-cheap-but-irrelevant rows (mostly drinks and desserts) drowned out the one real
match. Adding `search_query` + `category_hint` alone wasn't enough on the next run either:
the model guessed `lighter bites` and `big flavour bites` but not `bao buns` -- the category
the real match is actually in. All four (`bao buns`, `big flavour bites`, `gyoza`,
`lighter bites`) turn out to share the same `category_path` parent, `sides` -- this menu's
own closest thing to a starters section, just never labelled that anywhere. (Confusingly,
`sides` is *also* used as its own unrelated leaf category, but only under `gluten free`'s
menu section -- nothing to do with this group; a corpus vocabulary quirk in its own right.)
`expand_category_hint()` reads that sibling relationship straight from `category_path`, so
the two correct guesses now pull `bao buns` in too.

This call uses `UNDERSTAND_TEMPERATURE = 0.0` (deterministic) plus Groq's strict
`json_schema` response format. The schema does the heavy lifting regardless of
temperature -- it constrains the output's structure and every enum-typed field so the
model literally cannot return an invalid category or allergen name -- and
`temperature=0` additionally makes the free-form fields (`search_query`,
`price_max_gbp`, ...) as repeatable as this kind of extraction gets.

**Follow-up questions.** `understand_query()` takes an optional `context` -- the last
few turns rendered by `build_history_context()` (same as `app/agent/memory.py`), sent as
`Recent conversation so far: ... Guest's new message: ...`. The prompt tells the model to
use the prior turns only to resolve a pronoun or implicit reference ("what about the vegan
one?"), never to lift price, allergy or dietary wording from an earlier turn. With no history
the model sees the bare question, exactly as before. `search()` and `answer()` take a
`history` argument and pass it through. The same call also returns `resolved_question`, the new
message rewritten to stand on its own ("how many calories does it have?" becomes "how many
calories does the chicken katsu curry have?"); `build_user_prompt()` gives it to generation, and
nothing else uses it (rules `U-22`, `P-05`, `C-22`).

**Card clicks.** A click on a group or category card is answered with no understanding call at
all: `answer(..., browse={"group", "category"})` builds the understanding with the app's
`picked_browse()`, exactly as the app's graph does (rule `R-15`).

**Where the prompt comes from.** `UNDERSTAND_SYSTEM_PROMPT` is built by
`build_understand_system_prompt()` in `app/agent/prompts.py` -- the same function the app
calls -- with the two vocabularies this cell reads live from the collection (canonical
allergens, menu categories) appended, so the two can't describe the fields differently.
To change what the model is told, edit `app/agent/prompts.py`, not this notebook.

**Routing (added 2026-09-21).** `intent` is now one of `greeting`, `off_topic`,
`menu_browse`, `menu` or `faq`. The first three never reach retrieval or generation: a greeting
gets a fixed greeting, an off-topic message a fixed polite redirect, and a menu-browsing message
a list built from the catalog -- the groups, then a group's categories, then a category's items
-- with no hybrid search and no rerank (section 8). Two new fields, `browse_group` and
`browse_category`, carry which group or category the guest named or chose. A browse that also
states a requirement (a diet, an allergy, a price, calories, protein, non-alcoholic) is turned
into an ordinary `menu` search by the parser, because a listing would ignore it. The prompt also
describes the knowledge base to the model -- every field, every value of the limited-value
fields, the item types, and each item type's groups and categories -- rendered from the live
corpus by the same code the app uses. The rules are written up, with IDs, in
`docs/LLM_RULES.md`.

In [ ]:
import app.agent.understanding as app_understanding
from app.agent.memory import build_history_context
from app.agent.understanding import (
    UNDERSTAND_REASONING_EFFORT,
    UNDERSTAND_TEMPERATURE,
    UnderstandingResult,
    build_response_schema,
    build_understand_system_prompt,
    load_category_index,
    parse_understanding,
    picked_browse,
)

# How a message is understood -- the prompt (including the description of the knowledge base's
# structure), the response schema, the parsing and its routing safety net -- is the app's own
# code, imported rather than copied, so this notebook runs exactly what ships. Only the LLM call
# itself is this notebook's: call_llm() (section 2) also supports Gemini and the key pool.
# picked_browse() is the understanding of a card click (answer()'s `browse`), which the app
# builds with no model call at all.

# Read live from the collection by the function the app calls at startup: the menu's categories,
# their sibling groups, the alcohol-only categories, and the catalog of groups, categories and
# item names that menu browsing (section 8) is answered from.
CATEGORY_INDEX = load_category_index(kb)
CATALOG = CATEGORY_INDEX.catalog
MENU_CATEGORIES = CATEGORY_INDEX.categories
ALCOHOLIC_ONLY_CATEGORIES = CATEGORY_INDEX.alcoholic_only
print(
    f"catalog: {CATALOG.total_rows} rows, {len(CATALOG.groups)} menu groups, "
    f"{len(MENU_CATEGORIES)} categories, {sum(CATALOG.item_type_counts.values())} rows profiled"
)

UNDERSTAND_SYSTEM_PROMPT = build_understand_system_prompt(CATEGORY_INDEX)
RESPONSE_SCHEMA = build_response_schema(CATEGORY_INDEX)


def understand_query(question: str, context: str = "") -> UnderstandingResult:
    """Single LLM call: decide how the message is handled (greeting, off-topic, menu browsing, a
    dish or search question, or house policy) and extract the retrieval filters.

    `context` is optional recent-conversation text (build_history_context()) so a follow-up, or a
    choice from a list the assistant just sent, can be resolved. `question` itself, and every
    fallback, stays the guest's bare current message.
    """
    if not LLM_AVAILABLE:  # the app's deterministic no-key fallback: a plain search, no filters
        return app_understanding.understand_query(
            question, category_index=CATEGORY_INDEX, groq_client=None, model=UNDERSTAND_MODEL
        )
    user_message = f"{context}\n\nGuest's new message: {question}" if context else question
    resp = call_llm(
        UNDERSTAND_SYSTEM_PROMPT,
        user_message,
        model=UNDERSTAND_MODEL,
        response_schema=RESPONSE_SCHEMA,
        temperature=UNDERSTAND_TEMPERATURE,
        reasoning_effort=UNDERSTAND_REASONING_EFFORT,
    )
    # The app's own UnderstandingResult, so every app function it is passed to gets the type
    # it declares.
    return parse_understanding(resp["text"], resp["usage"], question, CATEGORY_INDEX)

## 4. Retrieval pipeline

The rest of `01`'s pipeline, unchanged in substance -- only `parse_constraints()` is gone,
replaced by two functions consuming `understand_query()`'s output: `build_filter()` (dietary +
price -> a Weaviate server-side filter, as before) and `build_search_text()` (new -- see
below). `FIELDS` still carries the two additions generation needs: `description` (the actual
grounding text, not just the vectorized `embedding_text`) and `kcal`.

`search()` now retrieves and reranks against `build_search_text(u)`, not the raw question --
`understand_query()`'s cleaned `search_query` plus its `category_hint` appended as plain
text. `search()`'s result dict carries `search_text` too, so it's visible in every trace
alongside `understanding`, not just used silently.

`rerank()` now also returns Cohere's own `meta.billed_units.search_units` for its call --
real billing data (one search unit per up to 100 documents), not an estimate, zero on the
hybrid-order fallback since nothing billable happened. This is the only embedding-side cost
this notebook can actually observe: the hybrid query's own vectorization of your question text
happens *inside* Weaviate's managed `text2vec_cohere` integration, and Weaviate's query
response never reports what that internal Cohere call cost -- there is no client-visible
number for it with this architecture. `show_answer()` (section 8) reports the rerank units
per question and running-session total on that basis, not a full embedding cost.

`rerank()` now paces itself under `COHERE_MAX_RPM` too, the same proactive-pacing idea section
2 already applies to the LLM side (Groq) -- previously only the LLM side had this, but
constraint relaxation below means one question can now trigger several `rerank()` calls
instead of exactly one, which raises the odds of a Cohere 429 the same way multiple LLM
calls did before `_pace_llm_call()` existed.

**Constraint relaxation** -- borrowed from a similar RAG assignment
(`C1M5_Assignment_Solve.ipynb`, which relaxes clothing-store filters like colour and category
when a search comes up too thin). If nothing clears `GATE` on the first attempt, `search()`
drops one `RELAXABLE_FIELDS` constraint at a time (`kcal_max`, `protein_min_g`,
`gluten_free_only`, `alcohol_free`, then `price_max_gbp` last) and retries, instead of
declining outright when a slightly-off-spec match exists. Adapted for a restaurant rather than
copied outright: `dietary` and `allergens_exclude` are **never** in `RELAXABLE_FIELDS` and
never get cleared -- the clothing example relaxes every filter including gender and category,
which is fine for "no exact colour match" but would be actively harmful here (silently
suggesting a non-vegan dish to a vegan guest, or one containing a stated allergen). The
returned `relaxed_fields` list feeds into section 7's `build_user_prompt()`, so the model is
told explicitly which constraint was dropped and states the dish's real figure instead of
implying a false match. `rerank_search_units` accumulates across every relaxation attempt,
not just the last one, since each retry is a genuine extra Cohere call.

**This is `app/retrieval.py` itself (2026-09-25).** Everything described above runs in the
app's `RetrievalTool`, imported rather than copied: the Weaviate client has explicit timeouts
(5s to connect, 15s per query), rerank calls share one keep-alive `httpx.Client`, `FIELDS`
carries every field a card shows, and `retrieve()`, `rerank()` and the rerank pacer are timed as
`weaviate`, `rerank` and `rerank_pace` (section 1.1). This notebook's `NotebookRetrievalTool`
only also keeps the rows of each hybrid query, so section 9 can show the candidates before the
rerank. A hit is a plain dict (`uuid`, `score`, `properties`), and a reranked hit is
`{"row", "rerank", "hybrid"}`, as in the app.

In [ ]:
from typing import TypedDict

from app.retrieval import (
    ALPHA,
    GATE,
    TOP_N,
    K,
    MenuRow,
    RerankHit,
    RetrievalTool,
    SearchResult,
    allergen_set,
    plist,
    pnum,
    pstr,
)

# Retrieval is app/retrieval.py itself, imported rather than copied: the server-side filter, the
# search text, the hybrid query, the allergen exclusion, the Cohere rerank and its pacer, the
# gate, constraint relaxation, and the unfiltered lookup behind the "a filter removed this dish"
# NOTE are all the app's RetrievalTool, so this notebook searches exactly as the app does. A hit
# is a plain dict (MenuRow: `uuid`, `score`, `properties`) and a reranked hit is
# {"row", "rerank", "hybrid"}, as in the app. pstr()/plist()/pnum() read a row's properties.


class NotebookRetrievalTool(RetrievalTool):
    """The app's RetrievalTool, also keeping the rows of every hybrid query it runs so a trace
    can show the candidates before the rerank (02's explain_pipeline()). The search itself is
    unchanged."""

    def __init__(self, kb, cohere_api_key: str) -> None:
        super().__init__(kb, cohere_api_key)
        self.queries: list[tuple[int, list[MenuRow]]] = []

    def retrieve(self, query: str, *, k: int = K, **kwargs) -> list[MenuRow]:
        rows = super().retrieve(query, k=k, **kwargs)
        self.queries.append((k, rows))
        return rows


class NotebookSearchResult(SearchResult):
    """The app's SearchResult plus what this notebook's traces show: the understanding it was
    run on, and the rows of the hybrid query search() settled on, before and after the allergen
    exclusion."""

    understanding: UnderstandingResult
    retrieved_objects: list[MenuRow]
    kept_objects: list[MenuRow]


# One tool for the whole kernel session, as the app keeps one per process: it holds the Cohere
# pacing state and a keep-alive connection pool. Re-running this cell closes the previous one.
_previous_retrieval_tool = globals().get("retrieval_tool")
if _previous_retrieval_tool is not None:
    with contextlib.suppress(Exception):
        _previous_retrieval_tool.close()
retrieval_tool = NotebookRetrievalTool(kb, COHERE_KEY)
retrieve = retrieval_tool.retrieve


def search(
    question: str,
    gate: float = GATE,
    history: list[HistoryTurn] | None = None,
    understanding: UnderstandingResult | None = None,
) -> NotebookSearchResult:
    """The app's RetrievalTool.search() on `understanding`, understanding the question first
    when no `understanding` is given (answer() routes on it before searching).

    `history` (prior turns as [{"question", "answer"}]) is only used to resolve a follow-up
    during understanding -- see build_history_context(). Returns the app's SearchResult plus
    three keys for this notebook's traces: `understanding`, and `retrieved_objects` /
    `kept_objects`, the rows of the hybrid query search() settled on, before and after the
    allergen exclusion. Nothing downstream of search() (the prompt, generation, the cards)
    reads those three.
    """
    if understanding is None:
        with timed("understand"):
            understanding = understand_query(question, context=build_history_context(history or []))
    retrieval_tool.queries = []
    result = retrieval_tool.search(understanding, gate=gate)
    # The last query for the full candidate count is the one search() settled on (relaxation
    # can run several); a k=1 query after it is the unfiltered lookup behind the NOTE.
    retrieved = next((rows for k, rows in reversed(retrieval_tool.queries) if k == K), [])
    excluded = set(result["excluded"])
    kept = [row for row in retrieved if not (allergen_set(row) & excluded)]
    return NotebookSearchResult(
        **result, understanding=understanding, retrieved_objects=retrieved, kept_objects=kept
    )


def line(row: MenuRow) -> str:
    """One-line summary of a row: type, name, category, price, dietary tags."""
    price = pnum(row, "price_gbp")
    price_s = f"  £{price:.2f}" if price is not None else ""
    diet = plist(row, "dietary_tags")
    diet_s = f"  {diet}" if diet else ""
    name, category = pstr(row, "name"), pstr(row, "category")
    return f"[{pstr(row, 'item_type')}] {name} <{category}>{price_s}{diet_s}"

## 5. Tone policy

Two levers now, working together, both picked from the understanding step's intent: a tone
instruction (what to say) and a real `temperature` (how much to vary phrasing). This section
used to be "Temperature policy" (`TEMP_MENU = 0.2` / `TEMP_FAQ = 0.8`) before this notebook's
Gemini backend, then became tone-only once `gemini-3.8-flash` turned out to silently ignore
`temperature`/`top_p`/`top_k` entirely (confirmed against Google's own docs and forum). Groq's
`openai/gpt-oss-120b` genuinely applies `temperature` -- confirmed directly in this notebook's
Phase 1 smoke test (`temperature=0` returned the identical sentence three times in a row;
`temperature=1.8` varied every time) -- so both levers are back, and they control different
things: the instruction shapes *content*, the temperature shapes *phrasing variability*.

- **Menu** -- precise and literal tone, `temperature=0.2`. Price, allergens, and nutrition are
  facts pulled from `CONTEXT`; the instruction tells the model to stay close to `CONTEXT`'s
  exact wording rather than paraphrase a number or an allergen into something wrong, and the
  low temperature keeps phrasing close to the same safe wording call to call.
- **FAQ** -- warm and conversational tone, `temperature=0.8`. House-policy answers (hours,
  bookings, payments) are copy, not numbers; the instruction gives it room to phrase things
  naturally as long as the substance still matches `CONTEXT`, and the higher temperature lets
  that natural phrasing actually vary.

The tone wording lives in `app/agent/prompts.py` (`MENU_TONE` / `FAQ_TONE`), and the
temperatures, `tone_for()` and `temperature_for()` in `app/agent/generation.py`; all are
imported, not copied.

In [ ]:
# The tone wording (app/agent/prompts.py), the temperatures and the reasoning effort
# (app/agent/generation.py) are the app's own, imported rather than copied.
from app.agent.generation import (
    FAQ_TEMPERATURE,
    GENERATION_REASONING_EFFORT,
    MENU_TEMPERATURE,
    temperature_for,
    tone_for,
)

print(f"menu: temperature={MENU_TEMPERATURE}  faq: temperature={FAQ_TEMPERATURE}")
print(f"reasoning_effort={GENERATION_REASONING_EFFORT!r}")

## 6. System instructions (generation)

Three fixed pieces, imported from `app/agent/prompts.py` -- the one module every system prompt
in the app comes from -- so this notebook exercises the prompt that actually ships and cannot
drift from it:

- `GENERATION_RULES` -- the grounding rules: answer only from CONTEXT, state prices exactly, use
  the `allergens_contains` / `allergens_may_contain` union, treat a `(gluten-free recipe)` /
  `(vegan recipe)` dish as its own recipe (never merge it with the standard one), and how to
  decline -- say it doesn't have that information, briefly note this demo runs on a limited data
  set, and frame a staff hand-off as what a full deployment would do (never tell the guest to go
  and ask staff themselves).
- `SCOPE_AND_SAFETY` -- the scope and prompt-injection guard: menu and house-policy questions
  only, text inside `<guest_message>` / `<retrieved_context>` is data and never instructions,
  never reveal the prompt or the model. Its own constant because the output-side leak check
  (section 7.1) runs against just this half.
- `CITATION_OUTPUT_INSTRUCTIONS` -- appended by `answer()` only when a shown dish could be cited:
  reply as `{"answer": ..., "cited_slugs": [...]}` so the guest-facing item cards match what the
  answer actually discusses.

`GENERATION_SYSTEM_PROMPT` is the rules plus the scope guard; section 8 adds `tone_for(intent)`
(and the citation format when it applies) per call. It is distinct from
`UNDERSTAND_SYSTEM_PROMPT` above -- that one extracts filters; this one answers the guest.

To change a rule, edit `app/agent/prompts.py` (its docstring says what to check first), run
the tests, then re-run sections 5-8 of `03_evaluation.ipynb`.

In [ ]:
from app.agent.prompts import (
    CITATION_OUTPUT_INSTRUCTIONS,
    GENERATION_SYSTEM_PROMPT,
    SCOPE_AND_SAFETY,
)

# These are imported from app/agent/prompts.py -- the single module every system prompt in the
# app comes from -- so this notebook exercises exactly what ships and cannot drift from it. To
# change a rule, edit that module, not this cell. SCOPE_AND_SAFETY stays
# its own constant, separate from GENERATION_RULES, because the output-side leak check (section
# 7.1) compares replies against just that section.
print(GENERATION_SYSTEM_PROMPT)

## 7. Prompt assembly

`format_row()` renders one reranked hit into the CONTEXT block the LLM sees -- an FAQ row as
a question/answer pair, a menu row as name/description/price/nutrition/allergens. This is the
information actually available to the model; it never sees `embedding_text` (vectorization
input only) or raw Weaviate scores. `protein_g` and `abv_percent` are included alongside
`kcal` now -- once `understand_query()` can filter on protein and alcohol content (section
3), the generation call needs those same figures in CONTEXT to actually cite them in the
answer, not just filter silently on numbers the guest never sees confirmed back.

`build_user_prompt()` also takes `relaxed_fields` (section 4) -- when `search()` had to drop a
constraint to find any answer at all, that has to be visible to the model as plain instruction
text, not just internal bookkeeping, or it has no way to know the dish it's about to describe
doesn't actually meet every part of what the guest asked for.

**Imported from the app, not copied.** `format_row()`, `build_context()`, `citable_slugs()`
and `build_user_prompt()` are `app/agent/generation.py`'s own. `build_user_prompt()` wraps the
guest's message and the CONTEXT in `<guest_message>` / `<retrieved_context>` blocks (the
delimiting `SCOPE_AND_SAFETY` refers to) and, when the message refers back to an earlier turn,
adds what it means (`resolved_question`, rule `P-05`); `format_row()` puts each menu item's
`slug` in its header line, the identifier the model cites. The item cards come from
`app/agent/cards.py`'s `cards_for_answer()`: every retrieved dish the answer names in full or
by a shortened name, or cites, and never a dish a filter screened out (rules `R-13`, `C-21`).
Each card carries the dish's whole guest-facing row (`C-26`).

In [ ]:
# Prompt assembly and the item cards are the app's own code, imported rather than copied:
#   format_row() / build_context()  the CONTEXT rows the model sees
#   citable_slugs()                 the dishes the model may cite (retrieved, with an image)
#   build_user_prompt()             the <guest_message> / <retrieved_context> blocks and the NOTEs,
#                                   including what a follow-up refers back to (P-05)
#   cards_for_answer()              the answer's item cards: every retrieved dish it names or
#                                   cites, never a dish a filter screened out (R-13, C-21)
from app.agent.cards import cards_for_answer
from app.agent.generation import (
    build_context,
    build_user_prompt,
    citable_slugs,
)

## 7.1 Streaming, citations and the leak check

Imported from `app/agent/generation.py`, not copied. Answer generation is streamed, which raises three
problems this cell solves:

- **Decoding.** When a shown dish can be cited, the model replies with a JSON object, so the raw
  stream is JSON, not prose. `AnswerStreamDecoder` pulls the `"answer"` string out of it
  incrementally (an escape split across chunks waits for its remaining characters), so
  `cited_slugs` is never shown to the guest.
- **Leak check.** `contains_system_prompt_leak()` flags a reply that repeats 8+ consecutive words
  of `SCOPE_AND_SAFETY`. Streaming would show a leak before a post-hoc check could run, so
  `LeakHoldback` releases text a few words behind the model and stops the stream the moment a
  leak is flagged; the guest gets `SAFE_FALLBACK_REPLY` instead.
- **Authoritative parse.** What streams is only a preview. `parse_generation_reply()` parses the
  complete reply (tolerating a stray code fence), filters `cited_slugs` to the retrieved
  candidates, and `generate_answer()` (section 8) falls back to the decoded text if the model
  didn't produce the JSON at all, keeping any `cited_slugs` it had already written
  (`salvage_cited_slugs()`, as the app does).

In [ ]:
# Streaming, the leak check and the reply parse are app/agent/generation.py's own code, imported
# rather than copied. generate_answer() (next section) runs them in the same order as the app's
# _generate_answer().
from app.agent.generation import (
    MALFORMED_REPLY,
    SAFE_FALLBACK_REPLY,
    AnswerStreamDecoder,
    LeakHoldback,
    contains_system_prompt_leak,
    parse_generation_reply,
    salvage_cited_slugs,
)

## 8. End-to-end answer function

`answer()` is the full path: `search()` (LLM understanding + retrieval) -> `tone_for()` from
the understanding step's intent, appended to `GENERATION_SYSTEM_PROMPT` -> prompt assembly ->
`generate_answer()` (a streamed `stream_llm()` call plus the leak check and citation parse
of section 7.1) -- and returns every intermediate artifact, not just the final text. `show_answer()` prints all of
it: what the understanding call extracted, the retrieval verdict, the exact system and user
prompts sent to the generation call, the answer, and what this question actually cost on
**both** APIs the pipeline touches:

- **LLM (Groq)** -- understanding and generation token counts, separately, from each call's
  own `usage` block (not estimated). Both calls use `openai/gpt-oss-120b` today
  (`UNDERSTAND_MODEL`/`GENERATION_MODEL`), but are tracked and configured separately since
  they're free to diverge later.
- **Embedding (Cohere)** -- the rerank call's `search_units`, real billing data from section
  4's `rerank()`. The hybrid query's own vectorization of your question text is *not*
  included -- it happens inside Weaviate's managed integration and Weaviate never reports
  what that internal Cohere call cost, so there is nothing to read for it from this notebook.

Both roll up into `SESSION_USAGE`, so a whole "Run All" shows its cumulative consumption
across every test cell, not just the last question's.

**Latency.** `answer()` runs inside one `track_timings()` turn, so its result carries
`timings` (ms per stage) and `show_answer()` prints them under `--- LATENCY ---`: end-to-end
`turn`, each stage's share, time spent waiting (our Cohere pacer, retry backoff) rather than
working, time to the model's first token and to the first word the guest can see, and
streaming speed. Section 1.1 explains each figure; section 11 turns them into p50/p95.

**Routing.** `answer()` runs `understand_query()` first. When the intent is `greeting`,
`off_topic` or `menu_browse` it returns a direct reply built by the app's `direct_answer()`
(`route: "direct"` in the result): one LLM call, no Weaviate query, no rerank and no generation
call, so `generate`, `weaviate` and `rerank` are absent from its timings. A list of groups or
categories also returns `choices` (the reply cut around one picture card per name), and a
category's item listing returns `intro` / `outro` around its item cards, as the app's `/chat`
response does. Otherwise it continues exactly as before (`route: "search"`).

**Kept in step with `app/agent/graph.py` (2026-09-25).** `answer()` passes `resolved_question`
to `build_user_prompt()`, takes its item cards from `cards_for_answer()`, keeps citations
salvaged from a broken JSON reply, and takes a card click as `browse={"group", "category"}`
-- answered with no understanding call and marked `card_click` in the returned history, as in
the app. `show_answer(question, history, browse)` prints the resolved question and any
picture cards too.

In [ ]:
from app.agent.browse import DIRECT_INTENTS, direct_answer, is_known_pick
from app.agent.cards import Choices, CitedItem


class GenerationResult(TypedDict):
    """What generate_answer() returns: the authoritative reply, the citations, the call's token
    usage and any server-side durations the provider reported."""

    reply: str
    cited_slugs: list[str]
    usage: Usage
    provider_timings: dict[str, float]


SESSION_USAGE = {"questions": 0, "total_tokens": 0, "rerank_search_units": 0}
# Every turn show_answer() has run, for a session-wide latency_report() (section 1.1).
SESSION_TIMINGS: list[dict[str, float]] = []


def generate_answer(
    system_prompt: str,
    user_prompt: str,
    *,
    model: str,
    temperature: float,
    candidate_slugs: list[str],
    turn_started: float,
    on_delta: Callable[[str], None] | None = None,
) -> GenerationResult:
    """Run the generation call as a stream -> {"reply", "cited_slugs", "usage",
    "provider_timings"}. The same control flow as app/agent/graph.py's _generate_answer().

    Every chunk of guest-visible text is handed to `on_delta` (the app hands it to LangGraph's
    stream writer; pass `print` to watch an answer appear live). The returned reply is always
    the authoritative one, parsed from the complete stream -- what was streamed is only a
    live preview of it.

    Output-side check: defense-in-depth behind the system prompt's own "never reveal yourself"
    instruction, not a replacement for it. Checked against SCOPE_AND_SAFETY specifically, not
    the full system prompt -- GENERATION_RULES deliberately instructs content that's supposed to
    reach the guest almost verbatim (e.g. the demo/limited-data decline wording), so scanning
    the reply against it produces false positives. Streaming would otherwise show a leak before
    this check could run, so LeakHoldback releases text a few words behind the model and stops
    the stream the moment a leak is flagged.

    Latency: `first_token` is stamped when the model's first text piece arrives (measured from
    the start of this stage, so it includes opening the stream and any retry backoff),
    `first_delta` when the first word is actually shown (measured from `turn_started`), and
    `stream` / `gen_chunks` describe the streaming itself (section 1.1).
    """
    json_mode = bool(candidate_slugs)
    decoder = AnswerStreamDecoder(json_mode=json_mode)
    guard = LeakHoldback(SCOPE_AND_SAFETY)
    raw: list[str] = []
    chunks = 0
    first_chunk_at = last_chunk_at = 0.0

    def show(visible: str) -> None:
        if visible:
            mark("first_delta", turn_started)
            if on_delta is not None:
                on_delta(visible)

    generate_started = time.perf_counter()
    with timed("generate"):
        # No response_format on purpose: Groq only streams tokens when none is set (see
        # parse_generation_reply()); the reply shape comes from the system prompt instead.
        stream = stream_llm(
            system_prompt,
            user_prompt,
            model=model,
            temperature=temperature,
            reasoning_effort=GENERATION_REASONING_EFFORT,
        )
        try:
            for piece in stream:
                now = time.perf_counter()
                if chunks == 0:
                    mark("first_token", generate_started)
                    first_chunk_at = now
                last_chunk_at = now
                chunks += 1
                raw.append(piece)
                show(guard.push(decoder.feed(piece)))
                if guard.leaked:
                    break
        finally:
            stream.close()
    if chunks:
        record("gen_chunks", chunks)
        record("stream", (last_chunk_at - first_chunk_at) * 1000)

    def result(reply: str, cited: list[str]) -> GenerationResult:
        return {
            "reply": reply,
            "cited_slugs": cited,
            "usage": stream.usage,
            "provider_timings": stream.provider_timings,
        }

    if guard.leaked:
        return result(SAFE_FALLBACK_REPLY, [])

    text = "".join(raw).strip()
    cited_slugs: list[str] = []
    if json_mode:
        parsed = parse_generation_reply(text, candidate_slugs)
        if parsed is not None:
            reply, cited_slugs = parsed
        else:
            # The model didn't produce the requested JSON. Keep whatever is still usable, and
            # say so -- a rising rate of these means the prompt-only format has stopped holding.
            print("   [generation reply was not the requested JSON object]")
            # Citations written before the JSON broke still count, as in the app.
            cited_slugs = salvage_cited_slugs(text, candidate_slugs)
            if decoder.text:
                reply = decoder.text
            elif text.startswith(("{", "`")):
                reply = MALFORMED_REPLY
            else:
                reply = text
    else:
        reply = text
    if contains_system_prompt_leak(SCOPE_AND_SAFETY, reply):
        return result(SAFE_FALLBACK_REPLY, [])

    tail = guard.flush()
    if tail:
        show(tail)
    return result(reply, cited_slugs)


def _direct_search_result(understanding: UnderstandingResult) -> NotebookSearchResult:
    """A search() result for a message answered without a search, so the rest of the notebook
    (show_answer(), the evaluation's checks) can treat every answer alike."""
    return {
        "understanding": understanding,
        "relaxed_fields": [],
        "excluded_top_match": None,
        "search_text": "",
        "excluded": [],
        "retrieved": 0,
        "retrieved_objects": [],
        "kept": 0,
        "kept_objects": [],
        "ranked": [],
        "top": 0.0,
        "answerable": False,
        "rerank_search_units": 0,
    }


def answer(
    question: str,
    history: list[HistoryTurn] | None = None,
    on_delta: Callable[[str], None] | None = None,
    browse: dict | None = None,
) -> dict:
    """Full pipeline: understand -> either a direct reply (a greeting, an off-topic message or
    menu browsing: no search, no second model call) or search -> ground -> generate (streamed).

    Mirrors app/agent/graph.py's query -> respond, or query -> retrieve -> ground -> answer, and
    returns every intermediate artifact, not just the text. `history` is the prior turns as
    [{"question", "answer"}] (the result's own "history" is that plus this turn, ready to pass
    to the next call); leave it out for a single-turn question. `browse` is a card click,
    {"group", "category"} (category None for a group's card): a known one is answered as
    that browse with no understanding call, as the app does (R-15); `question` is then the
    message the click sends ("Show me <category> from <group>"). Everything runs inside one
    track_timings() turn, so the result carries a "timings" dict (ms per stage, section 1.1).
    """
    history = history or []
    turn_started = time.perf_counter()
    with track_timings() as timings, timed("turn"):
        pick = (
            browse
            if browse and is_known_pick(CATALOG, browse["group"], browse["category"])
            else None
        )
        if pick:
            # A card click names its group/category exactly: nothing to understand (R-15).
            understanding = picked_browse(question, pick["group"], pick["category"])
        else:
            with timed("understand"):
                understanding = understand_query(question, context=build_history_context(history))
        intent = understanding["intent"]
        direct = intent in DIRECT_INTENTS
        cards: list[CitedItem] = []  # item cards a direct reply supplies (a category listing)
        # A list of groups or categories cut around its picture cards (R-14), or a category's
        # item listing cut around its item cards (C-25); None on every other reply.
        choices: Choices | None = None
        intro: str | None = None
        outro: str | None = None
        ranked: list[RerankHit] = []
        result: NotebookSearchResult
        gen: GenerationResult
        if direct:
            result = _direct_search_result(understanding)
            tone, temperature, system_prompt = "", None, ""
            # There is no CONTEXT: say so in the prompt the judge is shown (03_evaluation).
            user_prompt = build_user_prompt(
                question,
                f"(none -- answered directly, without a search: intent {intent})",
                [],
                None,
            )
            shown = direct_answer(understanding, CATALOG)
            reply = shown.text
            cards = list(shown.cards)
            choices, intro, outro = shown.choices, shown.intro, shown.outro
            # What the app streams: only the sentence before a reply's cards, never the bullet
            # list those cards replace.
            preview = choices["intro"] if choices else intro or reply
            mark("first_delta", turn_started)
            if on_delta is not None:
                on_delta(preview)
            gen = {
                "reply": reply,
                "cited_slugs": [],
                "usage": zero_usage(),
                "provider_timings": {},
            }
        else:
            result = search(question, history=history, understanding=understanding)
            tone = tone_for(intent)
            temperature = temperature_for(intent)
            ranked = result["ranked"] if result["answerable"] else []
            candidate_slugs = citable_slugs(ranked)
            system_prompt = f"{GENERATION_SYSTEM_PROMPT}\n\n{tone}"
            if candidate_slugs:
                system_prompt += f"\n\n{CITATION_OUTPUT_INSTRUCTIONS}"
            context = build_context(ranked) if result["answerable"] else "(no confident match)"
            user_prompt = build_user_prompt(
                question,
                context,
                result["relaxed_fields"],
                result["excluded_top_match"],
                resolved_question=understanding["resolved_question"],
            )
            if not LLM_AVAILABLE:
                gen = {
                    "reply": "[no LLM key set -- skipping live call]",
                    "cited_slugs": [],
                    "usage": zero_usage(),
                    "provider_timings": {},
                }
            else:
                gen = generate_answer(
                    system_prompt,
                    user_prompt,
                    model=GENERATION_MODEL,
                    temperature=temperature,
                    candidate_slugs=candidate_slugs,
                    turn_started=turn_started,
                    on_delta=on_delta,
                )
    reply = gen["reply"]
    item_cards: list[CitedItem]
    if direct:
        item_cards = cards
    else:
        screened = result["excluded_top_match"]
        item_cards = cards_for_answer(
            reply,
            ranked=ranked,
            cited_slugs=gen["cited_slugs"],
            screened_out=screened["name"] if screened else None,
        )
    turn: HistoryTurn = {"question": question, "answer": reply}
    if pick:
        turn["card_click"] = True  # free, so the app does not count it towards the turn cap
    understand_usage = understanding["usage"]
    usage = {
        "understand": understand_usage,
        "generate": gen["usage"],
        "total_tokens": understand_usage["total_tokens"] + gen["usage"]["total_tokens"],
    }
    return {
        "question": question,
        "route": "direct" if direct else "search",
        "search": result,
        "intent": intent,
        "tone": tone,
        "temperature": temperature,
        "system_prompt": system_prompt,
        "user_prompt": user_prompt,
        "answer": reply,
        "cited_slugs": gen["cited_slugs"],
        "cited_items": item_cards,
        "choices": choices,
        "intro": intro,
        "outro": outro,
        "usage": usage,
        "timings": timings,
        "provider_timings": gen["provider_timings"],
        "history": [*history, turn],
    }


def show_answer(
    question: str, history: list[HistoryTurn] | None = None, browse: dict | None = None
) -> dict:
    """Run answer() and print the full trace: understanding, retrieval, prompts, reply, latency."""
    r = answer(question, history=history, browse=browse)
    s = r["search"]
    u = s["understanding"]
    usage = r["usage"]
    rerank_units = s["rerank_search_units"]
    SESSION_USAGE["questions"] += 1
    SESSION_USAGE["total_tokens"] += usage["total_tokens"]
    SESSION_USAGE["rerank_search_units"] += rerank_units
    SESSION_TIMINGS.append(r["timings"])
    verdict = "ANSWERABLE" if s["answerable"] else "NO CONFIDENT MATCH"
    print(f"q: {question!r}")
    print(
        f"   understanding: intent={u['intent']}  dietary={u['dietary']}  "
        f"price_max_gbp={u['price_max_gbp']}  allergens_exclude={u['allergens_exclude'] or '-'}  "
        f"category_hint={u['category_hint'] or '-'}"
    )
    print(
        f"   gluten_free_only={u['gluten_free_only']}  kcal_max={u['kcal_max']}  "
        f"protein_min_g={u['protein_min_g']}  alcohol_free={u['alcohol_free']}"
    )
    if u["resolved_question"] != question:
        print(f"   resolved_question: {u['resolved_question']!r}")
    if browse:
        print(f"   card click: {browse}  -- no understanding call")
    if r["route"] == "direct":
        print(
            f"   route: answered directly (intent={r['intent']})  "
            f"browse_group={u['browse_group']}  browse_category={u['browse_category']}  "
            "-- no search, no generation call"
        )
    else:
        print(f"   search_text: {s['search_text']!r}")
        print(
            f"   retrieval={verdict} (top rr {s['top']:.3f})  "
            f"retrieved {s['retrieved']} -> kept {s['kept']}"
            + (f"  relaxed={s['relaxed_fields']}" if s["relaxed_fields"] else "")
        )
        tone_label = "faq" if r["intent"] == "faq" else "menu"
        print(
            f"   generation tone={tone_label}  temperature={r['temperature']}  "
            f"reasoning_effort={GENERATION_REASONING_EFFORT!r}"
        )
    print(
        f"   LLM usage: understand {usage['understand']['total_tokens']} tok + "
        f"generate {usage['generate']['total_tokens']} tok = {usage['total_tokens']} tok "
        f"for this question  (session so far: {SESSION_USAGE['total_tokens']} tok over "
        f"{SESSION_USAGE['questions']} questions)"
    )
    print(
        f"   embedding usage: rerank {rerank_units} search unit(s) this question  "
        f"(session so far: {SESSION_USAGE['rerank_search_units']}); query vectorization runs "
        f"inside Weaviate and isn't reported back to the client -- not counted here"
    )
    if r["route"] == "search":
        print("\n--- SYSTEM PROMPT (generation) ---")
        print(r["system_prompt"])
        print("\n--- USER PROMPT ---")
        print(r["user_prompt"])
    print("\n--- ANSWER ---")
    print(r["answer"])
    cards = [item["name"] for item in r["cited_items"]]
    print(f"\n   cited_slugs={r['cited_slugs'] or '-'}  item cards shown: {cards or '-'}")
    if r["choices"]:
        pictures = [card["name"] for card in r["choices"]["cards"]]
        print(f"   picture cards shown in place of the list: {pictures}")
    print("\n--- LATENCY ---")
    print(format_timings(r["timings"], r["usage"]))
    if r["provider_timings"]:
        reported = ", ".join(f"{k}={v * 1000:,.0f} ms" for k, v in r["provider_timings"].items())
        print(f"  provider-reported (generate call): {reported}")
    print()
    return r

## 9. Step-by-step pipeline walkthrough

`explain_pipeline()` runs the exact same `answer()` used everywhere else in this notebook --
no new logic, only a different presentation -- and renders every stage of the pipeline as a
numbered, human-readable walkthrough: what each stage was given, what it did, and what it
produced. `show_answer()` (section 8) stays as the compact trace for quick eyeballing; this is
the full narrative for understanding *why* a given answer came out the way it did.

In [ ]:
from IPython.display import Markdown, display


def _fmt_json(d: dict) -> str:
    return "```json\n" + json.dumps(d, indent=2, ensure_ascii=False) + "\n```"


def _describe_filter(u: dict) -> str:
    """Human-readable form of what build_filter(u) turns into a Weaviate filter."""
    clauses = []
    if u["dietary"]:
        clauses.append(f"`dietary_tags` contains `{u['dietary']}`")
    if u["price_max_gbp"] is not None:
        clauses.append(f"`price_gbp` <= {u['price_max_gbp']}")
    if u["gluten_free_only"]:
        clauses.append("`is_gluten_free_listed` is `true`")
    if u["kcal_max"] is not None:
        clauses.append(f"`kcal` <= {u['kcal_max']}")
    if u["protein_min_g"] is not None:
        clauses.append(f"`protein_g` >= {u['protein_min_g']}")
    if u["alcohol_free"]:
        clauses.append("`abv_percent` <= 0.5")
    return "; ".join(clauses) if clauses else "*none -- no hard filter applied, pure hybrid search*"


def _hybrid_score(row: MenuRow) -> float:
    return row["score"]


def _numbered(items: list[str]) -> str:
    return "\n".join(f"{i}. {text}" for i, text in enumerate(items, start=1))


DIRECT_INTENT_NOTES = {
    "greeting": (
        "Classified as a **greeting** (the whole message is a greeting or pleasantry). It is "
        "answered with the fixed greeting -- no search and no generation call."
    ),
    "off_topic": (
        "Classified as **off-topic** (not a greeting, and not about the restaurant's menu or "
        "house policy). It is answered with the fixed polite redirect -- no search and no "
        "generation call."
    ),
    "menu_browse": (
        "Classified as a **menu browse** (the guest wants to see how the menu is organised, "
        "with no other requirement). The reply is built from the catalog -- the groups, a "
        "group's categories, or a category's items -- with no hybrid search, no rerank and no "
        "generation call."
    ),
}


def explain_pipeline(question: str) -> dict:
    """Run the full pipeline via answer() and render it as a numbered, detailed walkthrough:
    every stage's exact input (including full prompts sent to the LLM), what it did, and its
    output. Purely a presentation layer over the same verified answer()/search() used by
    show_answer() -- no pipeline behavior changes here.
    """
    r = answer(question)
    s = r["search"]
    u = s["understanding"]

    steps: list[str] = []

    def step(title: str, body: str) -> None:
        steps.append(f"### Step {len(steps) + 1} -- {title}\n\n{body}")

    step("Guest asks a question", f"> {question}")

    understanding_view = {k: v for k, v in u.items() if k != "usage"}
    step(
        "Query understanding (LLM call)",
        f"**System prompt sent to `{UNDERSTAND_MODEL}`** "
        f"(`temperature={UNDERSTAND_TEMPERATURE}`, "
        f"`reasoning_effort={UNDERSTAND_REASONING_EFFORT!r}`):\n\n"
        f"```\n{UNDERSTAND_SYSTEM_PROMPT}\n```\n\n"
        f"**User message sent:**\n\n> {question}\n\n"
        f"**Output (parsed JSON):**\n\n{_fmt_json(understanding_view)}\n\n"
        f"*Cost: {u['usage']['total_tokens']} tokens "
        f"({u['usage']['prompt_tokens']} prompt + {u['usage']['completion_tokens']} completion).*",
    )

    if u["intent"] == "faq":
        intent_note = (
            "Classified as an **FAQ** question (house policy: hours, bookings, delivery, "
            "payments, gift cards, allergen-info pointers -- not menu content). "
            "`category_hint` stays empty by design for this intent, and generation will use "
            "**FAQ tone**."
        )
    elif u["intent"] == "menu":
        intent_note = (
            "Classified as a **menu** question (a dish, ingredient, price, nutrition value, "
            "dish comparison, or a general browse/availability question about the menu). "
            "Generation will use **menu tone**."
        )
    else:
        intent_note = DIRECT_INTENT_NOTES[u["intent"]]
    step("Identify how the message is handled", intent_note)

    if r["route"] == "direct":
        step(
            "Direct reply (no search, no generation call)",
            f"Browse group: `{u['browse_group']}`, browse category: `{u['browse_category']}`.\n\n"
            f"**Reply sent to the guest:**\n\n```\n{r['answer']}\n```",
        )
        step(
            "Latency breakdown",
            f"Measured with `timed()` around each stage of this one run (section 1.1):\n\n"
            f"```\n{format_timings(r['timings'], r['usage'])}\n```",
        )
        display(Markdown("\n\n---\n\n".join(steps)))
        print(
            f"\nTotal this question: {r['usage']['total_tokens']} tokens (understanding "
            "only), no Cohere calls."
        )
        return r

    step(
        "Build the retrieval query",
        f"**Search text** (dietary/price/allergy wording already stripped, `category_hint` "
        f'folded in as a soft signal): `"{s["search_text"]}"`\n\n'
        f"**Server-side filter:** {_describe_filter(u)}",
    )

    retrieved_lines = _numbered(
        [f"{line(o)}  (hybrid score: {_hybrid_score(o):.3f})" for o in s["retrieved_objects"]]
    )
    step(
        "Hybrid retrieval against `KnowledgeBase`",
        f"Weaviate hybrid search (`alpha={ALPHA}`, keyword + vector, top `{K}`) with the query "
        f"and filter above returned **{s['retrieved']} candidate row(s)**, in hybrid-score "
        f"order:\n\n{retrieved_lines}",
    )

    if s["excluded"]:
        dropped = [o for o in s["retrieved_objects"] if allergen_set(o) & set(s["excluded"])]
        dropped_lines = (
            _numbered([line(o) for o in dropped])
            if dropped
            else "*(none of the retrieved rows matched)*"
        )
        step(
            "Allergen exclusion",
            f"Rows whose `allergens_contains`/`allergens_may_contain` include any of "
            f"**{', '.join(s['excluded'])}** are dropped before ranking -- "
            f"kept **{s['kept']} of {s['retrieved']}**.\n\n**Dropped:**\n\n{dropped_lines}",
        )

    before_lines = _numbered(
        [f"{line(o)}  (hybrid score: {_hybrid_score(o):.3f})" for o in s["kept_objects"]]
    )
    after_lines = _numbered(
        [
            f"{line(h['row'])}  (rerank score: {h['rerank']:.3f}, hybrid score: {h['hybrid']:.3f})"
            for h in s["ranked"]
        ]
    )
    step(
        "Rerank (Cohere `rerank-v3.5`) -- before and after",
        f"**Before** -- the {len(s['kept_objects'])} kept candidate(s), still in hybrid-score "
        f"order:\n\n{before_lines}\n\n"
        f"**After** -- Cohere reranks all of them against the search text and keeps the top "
        f"`{TOP_N}`, reordered by actual relevance to the question rather than hybrid score "
        f"alone:\n\n{after_lines}",
    )

    relax_note = ""
    if s["relaxed_fields"]:
        relax_note = (
            "\n\nNo candidate cleared the gate on the first pass, so these constraints were "
            f"progressively relaxed, least essential first, and retrieval re-run: "
            f"**{', '.join(s['relaxed_fields'])}**."
        )
    if s["ranked"]:
        gate_input = _numbered(
            [f"{pstr(h['row'], 'name')}  (rerank score: {h['rerank']:.3f})" for h in s["ranked"]]
        )
    else:
        gate_input = "*(no candidates survived to be reranked)*"
    step(
        "Answerability gate",
        f"**Input** -- the {len(s['ranked'])} reranked item(s) from the previous step, with "
        f"their rerank scores:\n\n{gate_input}\n\n"
        f"**Score check:** top rerank score **{s['top']:.3f}** vs. required gate `{GATE}`."
        f"{relax_note}\n\n"
        f"**Output:** the question **{'cleared' if s['answerable'] else 'did NOT clear'}** the "
        f"gate -- retrieval is **{'ANSWERABLE' if s['answerable'] else 'NO CONFIDENT MATCH'}**.",
    )

    if s["answerable"]:
        item_names = [pstr(h["row"], "name") for h in s["ranked"]]
        context_body = (
            f"**{len(item_names)} row(s)** go into CONTEXT for generation:\n\n"
            + "\n".join(f"- {n}" for n in item_names)
        )
    else:
        context_body = (
            'No row cleared the gate, so CONTEXT is the literal string `"(no confident match)"` '
            "-- the system prompt's decline-don't-guess rule fires on exactly this."
        )
    if s["excluded_top_match"]:
        context_body += (
            f"\n\n**NOTE attached:** `{s['excluded_top_match']['name']}` was the closest "
            f"name match in the whole corpus but was excluded because it "
            f"{s['excluded_top_match']['reason']} -- generation is told this explicitly rather "
            "than silently substituting a different dish."
        )
    step("Assemble CONTEXT for generation", context_body)

    tone_label = (
        "FAQ -- warm, conversational" if r["intent"] == "faq" else "menu -- precise, literal"
    )
    step(
        "Determine tone and temperature",
        f"Using the intent classified in step 2 (**`{r['intent']}`**), generation is configured "
        f"with the **{tone_label}** tone at **`temperature={r['temperature']}`** and "
        f"**`reasoning_effort={GENERATION_REASONING_EFFORT!r}`**.\n\n"
        f"**Tone instruction applied** (appended to the fixed system prompt "
        f"below):\n\n> {r['tone']}",
    )

    gen_usage = r["usage"]["generate"]
    cards = ", ".join(f"`{i['slug']}`" for i in r["cited_items"]) or "none"
    step(
        "Generation (LLM call)",
        f"**System prompt sent to `{GENERATION_MODEL}`** (grounding rules + scope-and-safety "
        f"guard + the tone instruction from the previous step, plus the JSON output format "
        f"when a shown dish can be cited):\n\n```\n{r['system_prompt']}\n```\n\n"
        f"**User prompt sent** (question, what a follow-up refers back to, CONTEXT, any "
        f"NOTEs):\n\n"
        f"```\n{r['user_prompt']}\n```\n\n"
        f"**Output (generated answer):**\n\n{r['answer']}\n\n"
        f"**Item cards** (every retrieved dish the answer names or cites, never one a filter "
        f"screened out): {cards}\n\n"
        f"*Cost: {gen_usage['total_tokens']} tokens "
        f"({gen_usage['prompt_tokens']} prompt + {gen_usage['completion_tokens']} completion).*",
    )

    step(
        "Latency breakdown",
        f"Measured with `timed()` around each stage of this one run (section 1.1):\n\n"
        f"```\n{format_timings(r['timings'], r['usage'])}\n```",
    )

    display(Markdown("\n\n---\n\n".join(steps)))
    print(
        f"\nTotal this question: {r['usage']['total_tokens']} tokens "
        f"(understand {r['usage']['understand']['total_tokens']} + "
        f"generate {gen_usage['total_tokens']}), "
        f"{s['rerank_search_units']} Cohere rerank search unit(s)."
    )
    return r

## 10. Demo -- ask your own question

Edit `QUESTION` and re-run this cell as many times as you like -- it reuses the connection
opened in cell 1, so there's no need to re-run the notebook from the top.

In [ ]:
# QUESTION = "a vegan starter under £6"  # <- edit this, then re-run the cell
QUESTION = "give me drink without alcohol?"

_ = explain_pipeline(QUESTION)

# Multi-turn: pass a turn's "history" to the next question and a follow-up like
# "what about the vegan one?" is resolved against it, exactly as in the app.
# first = show_answer("what noodle dishes do you have?")
# _ = show_answer("what about a vegan one?", history=first["history"])

## 11. Latency benchmark

`show_answer()` times one question; this section times many, and reports what one number
can't: the spread. `benchmark_latency()` asks each question `runs` times, keeps the first
`warmup` turns apart as *cold*, and prints n / mean / p50 / p95 / max per stage for the rest
(section 1.1 defines every row).

Two caveats before reading the numbers:

- **It makes real calls** -- 2 LLM calls and at least one Cohere rerank per turn -- so it is off
  by default (`RUN_LATENCY_BENCHMARK = False`) and "Run All" won't spend quota by surprise.
- **`COHERE_MAX_RPM` paces the run.** At the default 15 it forces about 4s between rerank
  calls, which lands in `rerank_pace` and inflates `turn`. Set `COHERE_MAX_RPM` (section 4) to
  your key's real quota for production-shaped numbers, and read the `work` row (turn minus
  waiting) either way. The app's own numbers come from its `chat turn timings` log line
  (`scripts/latency_check.py --log`); the stage names match, so the two can be compared.

In [ ]:
# Set to True to run: this makes live LLM and Cohere calls (see the markdown above).
RUN_LATENCY_BENCHMARK = False

BENCHMARK_QUESTIONS = [
    "is there coffee?",  # menu, ingredient lookup
    "what time do you open?",  # faq
    "a vegan starter under £6",  # dietary + price filter + category hint
    "I'm allergic to peanuts, which noodle dishes can I have?",  # allergen exclude
]

if RUN_LATENCY_BENCHMARK:
    _ = benchmark_latency(BENCHMARK_QUESTIONS, runs=3, warmup=1)
else:
    print("benchmark skipped (RUN_LATENCY_BENCHMARK = False); session so far:")
    latency_report(SESSION_TIMINGS, "turns run through show_answer() this session")